# Lightweight YOLO detector for traffic signs plus no-sign fallback

This notebook treats each class folder in `../jpg` as one sign category. Because the dataset does not include bounding boxes, it creates a single full-image training box per image so you can still train a YOLO detector. That is a good starting point for a robot if the sign is centered and fills most of the frame; if you later label true boxes, swap them in and keep the rest of the pipeline.

If you add a `jpg/no_sign` folder with background images, the notebook will train it as a real `no sign` class. If you do not have background images yet, live inference still falls back to `no sign` whenever the model sees no confident detection.

The model defaults to `yolov8s` so you get stronger accuracy while still keeping the network compact for a robot. If you need maximum speed, switch `BASE_MODEL` to `yolov8n.pt`.

In [11]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU is visible to PyTorch. If you expect the 4070, check your CUDA-enabled PyTorch install and NVIDIA drivers.")


CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA GeForce RTX 4070 Laptop GPU


In [12]:
%pip install -q ultralytics pillow pyyaml

from __future__ import annotations

from pathlib import Path
import random
import shutil
from typing import Dict, List, Tuple

import torch
import yaml
from PIL import Image

# Tunable defaults for a robot: YOLOv8s is the better accuracy-first lightweight starting point.
# If you need maximum speed, change BASE_MODEL to 'yolov8n.pt'.
RANDOM_SEED = 42
VAL_RATIO = 0.2
BOX_MARGIN = 0.03
IMG_SIZE = 416
BASE_MODEL = "yolov8s.pt"
RUN_TRAINING = True
FORCE_GPU = True


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        has_images = (candidate / "jpg").exists()
        has_labels = (candidate / "Keras" / "lables.txt").exists() or (candidate / "Lighter" / "lables.txt").exists()
        if has_images and has_labels:
            return candidate
    raise FileNotFoundError("Could not find the jpg folder and a lables.txt file from the current notebook location.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
IMAGE_ROOT = PROJECT_ROOT / "jpg"
LABEL_FILE = (PROJECT_ROOT / "Keras" / "lables.txt") if (PROJECT_ROOT / "Keras" / "lables.txt").exists() else (PROJECT_ROOT / "Lighter" / "lables.txt")
YOLO_ROOT = PROJECT_ROOT / "yolo_signs"
DATA_YAML = YOLO_ROOT / "data.yaml"
NO_SIGN_FOLDER = "no_sign"
NO_SIGN_CLASS = "no sign"

CLASS_NAMES = [line.strip() for line in LABEL_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
FOLDER_TO_CLASS = {
    "roadClossed": "road closed",
    "stop": "stop",
    "oneway": "oneway",
}
HAS_NO_SIGN_IMAGES = (IMAGE_ROOT / NO_SIGN_FOLDER).exists()
if HAS_NO_SIGN_IMAGES:
    FOLDER_TO_CLASS[NO_SIGN_FOLDER] = NO_SIGN_CLASS
    if NO_SIGN_CLASS not in CLASS_NAMES:
        CLASS_NAMES = [*CLASS_NAMES, NO_SIGN_CLASS]
NAME_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

missing = [name for name in FOLDER_TO_CLASS.values() if name not in NAME_TO_ID]
if missing:
    raise ValueError(f"Labels file is missing these classes: {missing}. Found classes: {CLASS_NAMES}")

HAS_NO_SIGN_CLASS = NO_SIGN_CLASS in NAME_TO_ID


def list_images_for_class(folder_name: str) -> List[Path]:
    class_dir = IMAGE_ROOT / folder_name
    images = sorted([path for path in class_dir.glob("*.jpg") if path.is_file()])
    if not images:
        raise FileNotFoundError(f"No JPG files found in {class_dir}")
    return images


def yolo_box_line(class_id: int, margin: float = BOX_MARGIN) -> str:
    box_size = max(0.05, 1.0 - (2.0 * margin))
    return f"{class_id} 0.5 0.5 {box_size:.6f} {box_size:.6f}\n"


def split_images(images: List[Path], val_ratio: float, seed: int) -> Tuple[List[Path], List[Path]]:
    shuffled = images[:]
    random.Random(seed).shuffle(shuffled)
    if len(shuffled) <= 1:
        return shuffled, []

    val_count = max(1, int(round(len(shuffled) * val_ratio)))
    val_count = min(val_count, len(shuffled) - 1)
    val_images = shuffled[:val_count]
    train_images = shuffled[val_count:]
    return train_images, val_images


def prepare_yolo_dataset() -> Dict[str, int]:
    if YOLO_ROOT.exists():
        shutil.rmtree(YOLO_ROOT)

    for split_name in ("train", "val"):
        (YOLO_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (YOLO_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

    counts = {"train": 0, "val": 0}
    for folder_name, class_name in FOLDER_TO_CLASS.items():
        class_id = NAME_TO_ID[class_name]
        images = list_images_for_class(folder_name)
        train_images, val_images = split_images(images, VAL_RATIO, RANDOM_SEED + class_id)

        for split_name, split_subset in (("train", train_images), ("val", val_images)):
            for source_image in split_subset:
                target_stem = f"{folder_name}_{source_image.stem}"
                target_image = YOLO_ROOT / "images" / split_name / f"{target_stem}{source_image.suffix.lower()}"
                target_label = YOLO_ROOT / "labels" / split_name / f"{target_stem}.txt"

                shutil.copy2(source_image, target_image)
                target_label.write_text(yolo_box_line(class_id), encoding="utf-8")
                counts[split_name] += 1

    data_yaml = {
        "path": str(YOLO_ROOT),
        "train": "images/train",
        "val": "images/val",
        "names": CLASS_NAMES,
    }
    DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
    return counts


def build_model(weights: str = BASE_MODEL):
    from ultralytics import YOLO

    return YOLO(weights)


def train_model(weights: str = BASE_MODEL):
    from ultralytics import YOLO

    if FORCE_GPU and not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available to PyTorch. Install a CUDA-enabled PyTorch build so the 4070 can be used.")

    device = 0 if torch.cuda.is_available() else "cpu"
    if device != "cpu":
        print("Using GPU:", torch.cuda.get_device_name(0))

    model = YOLO(weights)
    results = model.train(
        data=str(DATA_YAML),
        epochs=120 if device != "cpu" else 20,
        imgsz=IMG_SIZE,
        batch=-1 if device != "cpu" else 4,
        device=device,
        optimizer="AdamW",
        lr0=0.003,
        patience=20 if device != "cpu" else 8,
        close_mosaic=15 if device != "cpu" else 5,
        freeze=10,
        cache=True if device != "cpu" else False,
        workers=8 if device != "cpu" else 2,
        amp=device != "cpu",
        project=str(PROJECT_ROOT / "runs"),
        name="sign_detector",
        exist_ok=True,
        pretrained=True,
        verbose=True,
    )
    return model, results


def predict_image(image_path: str | Path, weights_path: str | Path) -> object:
    from ultralytics import YOLO

    model = YOLO(str(weights_path))
    return model.predict(source=str(image_path), imgsz=IMG_SIZE, conf=0.25, device="cpu", verbose=False)


def predict_scene_label(image_path: str | Path, weights_path: str | Path, conf_threshold: float = 0.25) -> str:
    from ultralytics import YOLO

    model = YOLO(str(weights_path))
    results = model.predict(source=str(image_path), imgsz=IMG_SIZE, conf=conf_threshold, device="cpu", verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return NO_SIGN_CLASS

    best_idx = int(boxes.conf.argmax().item())
    class_id = int(boxes.cls[best_idx].item())
    return results[0].names[class_id]


def export_model(weights_path: str | Path, export_format: str = "torchscript") -> object:
    from ultralytics import YOLO

    model = YOLO(str(weights_path))
    return model.export(format=export_format)


counts = prepare_yolo_dataset()
print("Dataset prepared at:", YOLO_ROOT)
print("Class order:", CLASS_NAMES)
print("No-sign class available:", HAS_NO_SIGN_CLASS)
print("Image counts:", counts)
print("Training config file:", DATA_YAML)

if RUN_TRAINING:
    model, results = train_model()
    print("Training complete. Best weights should be under runs/sign_detector/weights/best.pt")


Note: you may need to restart the kernel to use updated packages.
Dataset prepared at: /media/noah/Shared/Robotics/model/yolo_signs
Class order: ['road closed', 'stop', 'oneway', 'no sign']
No-sign class available: True
Image counts: {'train': 261, 'val': 65}
Training config file: /media/noah/Shared/Robotics/model/yolo_signs/data.yaml
Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Ultralytics 8.4.45 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7808MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/noah/Shared/Robotics/model/yolo_signs/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fl